# Weber Posterior-Marginalized Feature Learning: Flip Set Analysis

Compares **BayesianSPO_Weber_Agent** variants against BayesianSPO baselines on Shortest Path 8x8.

Two marginalization modes:
- **MC** (original): Sample theta ~ N(mu, Sigma), compute SPO+ grad at each sample, average.
- **Hybrid** (new): SPO+ gradient at posterior mean (signal) + Weber's closed-form Sigma_n @ h (uncertainty correction). No MC sampling -- faster and deterministic.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import jax.numpy as jnp
import yaml
import time
from collections import defaultdict

from lets_plot import *
LetsPlot.setup_html()

In [2]:
# Load config
with open('../experiments/configs/experiment.yaml') as f:
    config = yaml.safe_load(f)

N_ROUNDS = 1000
N_SEEDS = 3
SEEDS = list(range(N_SEEDS))

DATASETS = ['knapsack_32']

ds_cfg = config['datasets']['knapsack_32']
N_ARMS = 32
CONTEXT_DIM = ds_cfg['context_dim']

print(f'Rounds: {N_ROUNDS}, Seeds: {N_SEEDS}')
print(f'Dataset: knapsack_32, Arms: {N_ARMS}, Context dim: {CONTEXT_DIM}')

Rounds: 1000, Seeds: 3
Dataset: knapsack_32, Arms: 32, Context dim: 5


## Setup: Flip Set & Tracking Utilities

In [3]:
from src.envs import create_env
from src.agents.base import pregenerate_episode
from src.agents.shared_context import (
    BayesianSPOAgent, BayesianMSEAgent, BayesianSPO_Weber_Agent,
)
from src.agents.shared_context.weber_bll import WeberBLLAgent
from src.agents.bayesian_utils import (
    compute_bayesian_predictions,
    compute_ucb_values,
    update_bayesian_stats,
)
import warnings
from scipy.stats import spearmanr, ConstantInputWarning


def compute_per_arm_std(features, A_inv):
    Ax = jnp.einsum('nij,j->ni', A_inv, features)
    variances = jnp.einsum('i,ni->n', features, Ax)
    return np.asarray(jnp.sqrt(jnp.maximum(variances, 0.0)))


def compute_flip_set(pred, true_vals, solver):
    pred_np = np.asarray(pred, dtype=np.float64)
    true_np = np.asarray(true_vals, dtype=np.float64)
    w_true = np.asarray(solver(true_np))
    w_spo = np.asarray(solver(2 * pred_np - true_np))
    flip_mask = np.abs(w_spo - w_true) > 0.5
    return flip_mask, int(flip_mask.sum()), w_true


def run_agent_with_flip_tracking(
    agent, episode, n_rounds, lr=0.01, full_feedback=False,
):
    import src.solver as _solver_module
    _solver_module._GUROBI_ENV = None

    rounds_data = episode['rounds_data']
    minimize = episode['minimize']

    rewards = []
    regrets = []
    cumulative_regret = []
    flip_set_sizes = []
    contested_sigma_sums = []
    contested_sigma_means = []
    contested_errors = []
    decision_correct = []
    contested_rank_corrs = []
    total_regret = 0.0

    for t in range(n_rounds):
        rd = rounds_data[t]
        X = rd['context']
        true_vals = rd['true_values']
        optimal_val = rd['optimal_value']

        h = agent._features(X)
        pred = compute_bayesian_predictions(h, agent.A_inv, agent.b_bayes)
        pred_np = np.asarray(pred)
        sigmas = compute_per_arm_std(h, agent.A_inv)

        flip_mask, flip_size, w_true = compute_flip_set(pred_np, true_vals, agent.solver)
        flip_set_sizes.append(flip_size)

        if flip_size >= 2:
            contested_sigma_sums.append(float(sigmas[flip_mask].sum()))
            contested_sigma_means.append(float(sigmas[flip_mask].mean()))
            contested_errors.append(float(np.mean(np.abs(pred_np[flip_mask] - true_vals[flip_mask]))))
            with warnings.catch_warnings():
                warnings.simplefilter('ignore', ConstantInputWarning)
                rc, _ = spearmanr(pred_np[flip_mask], true_vals[flip_mask])
            contested_rank_corrs.append(rc if not np.isnan(rc) else 0.0)
        else:
            contested_sigma_sums.append(0.0)
            contested_sigma_means.append(0.0)
            contested_errors.append(0.0)
            contested_rank_corrs.append(np.nan)

        selection = agent.act(X)
        selection = np.asarray(selection).flatten()
        agent_val = float(np.sum(true_vals * selection))

        if minimize:
            instant_regret = agent_val - optimal_val
        else:
            instant_regret = optimal_val - agent_val
        total_regret += instant_regret

        correct = (np.abs(selection - w_true) < 0.5).all()
        decision_correct.append(float(correct))

        rewards.append(agent_val if not minimize else -agent_val)
        regrets.append(instant_regret)
        cumulative_regret.append(total_regret)

        if full_feedback:
            observed = true_vals.copy()
        else:
            observed = np.zeros_like(true_vals)
            mask = selection > 0.5
            observed[mask] = true_vals[mask]

        agent.update(X, selection, observed, lr, full_feedback=full_feedback)

    return {
        'rewards': rewards,
        'regrets': regrets,
        'cumulative_regret': cumulative_regret,
        'final_regret': total_regret,
        'flip_set_sizes': flip_set_sizes,
        'contested_sigma_sums': contested_sigma_sums,
        'contested_sigma_means': contested_sigma_means,
        'contested_errors': contested_errors,
        'decision_correct': decision_correct,
        'contested_rank_corrs': contested_rank_corrs,
    }

## Agent Variants

In [4]:
def create_agents(env, seed, config, n_rounds):
    defaults = config['agent_defaults']
    hidden_dim = defaults['hidden_dim'] if env.context_dim > 20 else defaults['hidden_dim_small']
    lambda_reg = defaults['lambda_reg']
    minimize = getattr(env, 'minimize', False)
    solver = env.solve
    sign = -1.0 if minimize else 1.0

    spo_common = dict(
        context_dim=env.context_dim, n_arms=env.n_arms,
        weights=env.weights, capacity=env.capacity,
        hidden_dim=hidden_dim, lambda_reg=lambda_reg,
        minimize=minimize, solver=solver, seed=seed,
        mask_grad=True,
    )

    agents = {}

    # --- 2x2 ablation: loss (SPO+ vs MSE) x training (online vs phased) ---

    # SPO+ online (original BayesianSPO)
    agents['SPO-Online'] = BayesianSPOAgent(
        **spo_common, beta=sign * 1.0, decay_rounds=n_rounds,
    )

    # SPO+ phased (new: freeze features, batch retrain, replay)
    agents['SPO-Phased (K=50, E=5)'] = BayesianSPOAgent(
        **spo_common, beta=sign * 1.0, decay_rounds=n_rounds,
        phased=True, phase_length=50, phase_epochs=5,
    )

    # SPO+ phased with 1 epoch (same compute budget as online: 1 grad step per obs)
    agents['SPO-Phased (K=50, E=1)'] = BayesianSPOAgent(
        **spo_common, beta=sign * 1.0, decay_rounds=n_rounds,
        phased=True, phase_length=50, phase_epochs=1,
    )

    # MSE phased (WeberBLL, original)
    agents['WeberBLL-Phased (K=50, E=5)'] = WeberBLLAgent(
        **spo_common, beta=sign * 1.0, decay_rounds=n_rounds,
        phased=True, phase_length=50, phase_epochs=5,
    )

    # MSE phased with 1 epoch (compute-matched to online)
    agents['WeberBLL-Phased (K=50, E=1)'] = WeberBLLAgent(
        **spo_common, beta=sign * 1.0, decay_rounds=n_rounds,
        phased=True, phase_length=50, phase_epochs=1,
    )

    # MSE online
    agents['WeberBLL-Online'] = WeberBLLAgent(
        **spo_common, beta=sign * 1.0, decay_rounds=n_rounds,
        phased=False,
    )

    return agents

## Run Evaluation

In [5]:
all_results = {}

for dataset in DATASETS:
    print(f'\n{"="*60}')
    print(f'DATASET: {dataset}')
    print(f'{"="*60}')

    dataset_config = config['datasets'][dataset].copy()
    dataset_config.pop('env_type', None)
    lr = config['agent_defaults']['lr']

    results = defaultdict(list)

    for seed_idx, seed in enumerate(SEEDS):
        print(f'  Seed {seed_idx+1}/{len(SEEDS)} (seed={seed})', flush=True)

        env = create_env(dataset, seed=seed, **dataset_config)
        episode = pregenerate_episode(env, N_ROUNDS, full_feedback=False)

        env_for_agents = create_env(dataset, seed=seed, **dataset_config)
        agents = create_agents(env_for_agents, seed, config, N_ROUNDS)

        for agent_name, agent in agents.items():
            t0 = time.perf_counter()
            result = run_agent_with_flip_tracking(
                agent, episode, N_ROUNDS, lr=lr, full_feedback=False,
            )
            wall_time = time.perf_counter() - t0
            results[agent_name].append(result)

            final_flip = np.mean(result['flip_set_sizes'][-50:])
            print(f'    {agent_name:45s} regret={result["final_regret"]:8.1f}  '
                  f'flip_final={final_flip:.1f}  ({wall_time:.1f}s)', flush=True)

    all_results[dataset] = dict(results)


DATASET: knapsack_32
  Seed 1/3 (seed=0)
    SPO-Online                                    regret=  5536.9  flip_final=5.3  (4.6s)
    SPO-Phased (K=50, E=5)                        regret=  4734.0  flip_final=4.7  (95.4s)
    SPO-Phased (K=50, E=1)                        regret=  5000.9  flip_final=5.1  (21.4s)
    WeberBLL-Phased (K=50, E=5)                   regret=  4743.2  flip_final=4.8  (51.4s)
    WeberBLL-Phased (K=50, E=1)                   regret=  5114.5  flip_final=4.9  (15.5s)
    WeberBLL-Online                               regret=  5519.9  flip_final=5.8  (3.1s)
  Seed 2/3 (seed=1)
    SPO-Online                                    regret=  4734.6  flip_final=5.7  (5.7s)
    SPO-Phased (K=50, E=5)                        regret=  3436.7  flip_final=5.1  (99.4s)
    SPO-Phased (K=50, E=1)                        regret=  3950.2  flip_final=5.4  (21.5s)
    WeberBLL-Phased (K=50, E=5)                   regret=  3798.3  flip_final=5.2  (51.7s)
    WeberBLL-Phased (K=50, E=1)

## 1. Summary Table (Last 100 Rounds)

In [6]:
for dataset in DATASETS:
    print(f'\n--- {dataset} ---')
    rows = []
    for agent_name, res_list in all_results[dataset].items():
        final_regrets = [r['final_regret'] for r in res_list]
        mean_flip_last = [np.mean(r['flip_set_sizes'][-100:]) for r in res_list]
        mean_err = [np.mean(r['contested_errors'][-100:]) for r in res_list]
        mean_dec = [np.mean(r['decision_correct'][-100:]) for r in res_list]
        rank_vals = []
        for r in res_list:
            rc = np.array(r['contested_rank_corrs'][-100:])
            rc = rc[~np.isnan(rc)]
            rank_vals.append(np.mean(rc) if len(rc) > 0 else np.nan)

        rows.append({
            'Agent': agent_name,
            'Regret': f'{np.mean(final_regrets):.0f} +/- {np.std(final_regrets):.0f}',
            'Flip': f'{np.mean(mean_flip_last):.1f}',
            'Err': f'{np.mean(mean_err):.3f}',
            'Dec': f'{np.mean(mean_dec):.1%}',
            'Rank': f'{np.nanmean(rank_vals):.3f}',
        })
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    print('  Err=contested MAE | Dec=decision acc | Rank=Spearman on flip set (last 100)')


--- knapsack_32 ---
                      Agent       Regret Flip   Err  Dec  Rank
                 SPO-Online 5093 +/- 333  5.9 2.740 3.7% 0.342
     SPO-Phased (K=50, E=5) 3917 +/- 581  5.3 1.904 9.7% 0.333
     SPO-Phased (K=50, E=1) 4390 +/- 446  5.6 2.049 7.3% 0.354
WeberBLL-Phased (K=50, E=5) 4212 +/- 395  5.4 2.069 7.7% 0.377
WeberBLL-Phased (K=50, E=1) 4660 +/- 383  5.6 2.125 7.0% 0.425
            WeberBLL-Online 5370 +/- 331  6.3 2.765 5.0% 0.406
  Err=contested MAE | Dec=decision acc | Rank=Spearman on flip set (last 100)


## 2. Cumulative Regret Over Time

In [7]:
def build_regret_df(results, dataset):
    rows = []
    for agent_name, res_list in results.items():
        for seed_idx, res in enumerate(res_list):
            traj = np.array(res['cumulative_regret'])
            step = max(1, len(traj) // 200)
            for t in range(0, len(traj), step):
                rows.append({'Round': t, 'Cumulative Regret': traj[t],
                             'Agent': agent_name, 'Seed': seed_idx, 'Dataset': dataset})
    return pd.DataFrame(rows)

for dataset in DATASETS:
    reg_df = build_regret_df(all_results[dataset], dataset)
    agg = reg_df.groupby(['Round', 'Agent'])['Cumulative Regret'].mean().reset_index()
    p = (ggplot(agg, aes('Round', 'Cumulative Regret', color='Agent'))
         + geom_line(size=1)
         + ggtitle(f'Cumulative Regret: {dataset}')
         + theme(legend_position='bottom'))
    p.show()

## 3. Flip Set Size Over Time

In [8]:
def build_flip_df(results, dataset):
    rows = []
    for agent_name, res_list in results.items():
        for seed_idx, res in enumerate(res_list):
            flips = np.array(res['flip_set_sizes'])
            step = max(1, len(flips) // 200)
            for t in range(0, len(flips), step):
                rows.append({'Round': t, 'Flip Set Size': flips[t],
                             'Agent': agent_name, 'Seed': seed_idx, 'Dataset': dataset})
    return pd.DataFrame(rows)

for dataset in DATASETS:
    flip_df = build_flip_df(all_results[dataset], dataset)
    agg = flip_df.groupby(['Round', 'Agent'])['Flip Set Size'].mean().reset_index()
    p = (ggplot(agg, aes('Round', 'Flip Set Size', color='Agent'))
         + geom_line(size=1)
         + ggtitle(f'Flip Set Size: {dataset}')
         + theme(legend_position='bottom'))
    p.show()

## 4. Contested-Arm Uncertainty Over Time

In [9]:
def build_sigma_df(results, dataset, key='contested_sigma_sums', label='Contested Sigma Sum'):
    rows = []
    for agent_name, res_list in results.items():
        for seed_idx, res in enumerate(res_list):
            vals = np.array(res[key])
            step = max(1, len(vals) // 200)
            for t in range(0, len(vals), step):
                rows.append({'Round': t, label: vals[t],
                             'Agent': agent_name, 'Seed': seed_idx, 'Dataset': dataset})
    return pd.DataFrame(rows)

for dataset in DATASETS:
    sigma_df = build_sigma_df(all_results[dataset], dataset)
    agg = sigma_df.groupby(['Round', 'Agent'])['Contested Sigma Sum'].mean().reset_index()
    p = (ggplot(agg, aes('Round', 'Contested Sigma Sum', color='Agent'))
         + geom_line(size=1)
         + ggtitle(f'Contested-Arm Uncertainty: {dataset}')
         + theme(legend_position='bottom'))
    p.show()

## 5. Contested Error + Rank Correlation

In [10]:
for dataset in DATASETS:
    # Contested error
    err_df = build_sigma_df(all_results[dataset], dataset,
                            key='contested_errors', label='Contested MAE')
    agg = err_df.groupby(['Round', 'Agent'])['Contested MAE'].mean().reset_index()
    p = (ggplot(agg, aes('Round', 'Contested MAE', color='Agent'))
         + geom_line(size=1)
         + ggtitle(f'Contested-Arm MAE: {dataset}')
         + theme(legend_position='bottom'))
    p.show()

    # Rank correlation
    rank_df = build_sigma_df(all_results[dataset], dataset,
                             key='contested_rank_corrs', label='Rank Corr')
    agg = rank_df.groupby(['Round', 'Agent'])['Rank Corr'].mean().reset_index()
    p = (ggplot(agg, aes('Round', 'Rank Corr', color='Agent'))
         + geom_line(size=1)
         + ggtitle(f'Contested-Arm Rank Correlation: {dataset}')
         + theme(legend_position='bottom'))
    p.show()

## 6. Regret vs Flip Set Scatter

In [11]:
for dataset in DATASETS:
    rows = []
    for agent_name, res_list in all_results[dataset].items():
        for res in res_list:
            regrets = np.array(res['regrets'])
            flips = np.array(res['flip_set_sizes'])
            # Smooth with rolling window
            w = 50
            for t in range(w, len(regrets)):
                rows.append({
                    'Mean Regret': np.mean(regrets[t-w:t]),
                    'Mean Flip Size': np.mean(flips[t-w:t]),
                    'Agent': agent_name,
                })
    scat_df = pd.DataFrame(rows)
    p = (ggplot(scat_df, aes('Mean Flip Size', 'Mean Regret', color='Agent'))
         + geom_point(alpha=0.3, size=1)
         + ggtitle(f'Regret vs Flip Set Size (smoothed): {dataset}')
         + theme(legend_position='bottom'))
    p.show()

## 7. T₀ Collapse Time

In [12]:
def estimate_T0(flip_sizes, n_arms, threshold_frac=0.1, window=50):
    threshold = threshold_frac * n_arms
    flips = np.array(flip_sizes)
    for t in range(window, len(flips)):
        if np.mean(flips[t-window:t]) <= threshold:
            return t - window
    return len(flips)  # never collapsed

for dataset in DATASETS:
    print(f'\n--- {dataset}: T_0 estimates ---')
    for agent_name, res_list in all_results[dataset].items():
        t0s = [estimate_T0(r['flip_set_sizes'], N_ARMS) for r in res_list]
        print(f'  {agent_name:45s}  T_0 = {np.mean(t0s):.0f} +/- {np.std(t0s):.0f}')


--- knapsack_32: T_0 estimates ---
  SPO-Online                                     T_0 = 1000 +/- 0
  SPO-Phased (K=50, E=5)                         T_0 = 1000 +/- 0
  SPO-Phased (K=50, E=1)                         T_0 = 1000 +/- 0
  WeberBLL-Phased (K=50, E=5)                    T_0 = 1000 +/- 0
  WeberBLL-Phased (K=50, E=1)                    T_0 = 1000 +/- 0
  WeberBLL-Online                                T_0 = 1000 +/- 0
